# Twisted Parallel Particle Filter — 1D Linear-Gaussian SSM

This notebook walks through the **twisted parallel particle filter** on a 1-D linear-Gaussian state-space model (SSM). It is a self-contained demonstration of the algorithm from:

> *Parallelizing MCMC Across the Sequence Length* — Zoltowski et al., NeurIPS 2025

---

## What problem are we solving?

We observe a noisy time series $y_{1:T}$ and want to estimate the hidden state sequence $x_{1:T}$. This is called **smoothing** — computing $p(x_{1:T} \mid y_{1:T})$.

The standard approach is a **particle filter**, which approximates the posterior with $N$ weighted sample trajectories (particles). The catch: a standard particle filter is *sequential* — each step depends on the previous one, so it takes $O(T)$ sequential work.

The key innovation here is making the particle filter **parallel in $T$** by using:

1. **Twisted proposals** — informed proposals that keep particle weights balanced without resampling. This decouples particles from each other, so each of the $N$ trajectories is an *independent* recurrence.
2. **DEER** (Differentiated Evaluation via Evaluation Reverse) — a parallel Newton solver that evaluates all $T$ steps of each recurrence simultaneously in $O(\log T)$ depth via an associative scan.

---

## Model

$$x_0 \sim \mathcal{N}(0,\, 1)$$
$$x_t = \phi\, x_{t-1} + \sigma_q\,\varepsilon_t, \qquad \varepsilon_t \sim \mathcal{N}(0,1)$$
$$y_t = x_t + \sigma_r\,\eta_t, \qquad \eta_t \sim \mathcal{N}(0,1)$$

Because this model is linear and Gaussian, the exact posterior $p(x_{1:T} \mid y_{1:T})$ is available in closed form via the Kalman filter. We use it as a **gold standard** to measure how well the particle filter is doing.

---

## Roadmap

| Step | What | Why |
|------|------|-----|
| 1 | Define the Kalman filter | Computes the exact posterior — used to initialize twist parameters and as ground truth |
| 2 | Define twisted step & log-weight | The proposal distribution and correction weights |
| 3 | Simulate data | Generate $x_{1:T}$ and $y_{1:T}$ from the model |
| 4 | Run Kalman filter | Get exact posterior means/variances |
| 5 | Build the particle filter object | Wire up proposal + weight function |
| 6 | Sequential twisted PF | Baseline: $O(T)$ sequential depth |
| 7 | **Parallel twisted PF (DEER)** | Main result: $O(\log T)$ depth on GPU |
| 8 | Importance-weighted means | Correct for proposal bias |
| 9 | Classical baselines | Bootstrap PF + FFBSi for comparison |

## Setup

We suppress the Metal/XLA startup warnings (harmless but noisy), then import JAX and the two key modules:
- `src.particle_filter` — contains `TwistedParallelParticleFilter`, the main algorithm
- `src.baselines` — standard bootstrap PF and FFBSi smoother for comparison

The printout confirms which JAX version and hardware backend (Metal GPU on M1) are active.

In [ ]:
import os, logging, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress Metal/XLA C++ logs
logging.getLogger("absl").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

import time
import jax
jax.config.update("jax_default_matmul_precision", "highest")
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, '..')  # so 'src' is importable from examples/

from src.particle_filter import TwistedParallelParticleFilter
from src.baselines import bootstrap_pf, ffbsi_smoother

print(f"JAX version: {jax.__version__}")
print(f"Devices:     {jax.devices()}")

## Model parameters and problem size

| Symbol | Value | Role |
|--------|-------|------|
| $\phi$ | 0.9 | State auto-correlation. Close to 1 → slowly varying latent state |
| $\sigma_q$ | 0.5 | Process noise std. Smaller → smoother trajectory |
| $\sigma_r$ | 1.0 | Observation noise std. Larger than $\sigma_q$ → noisy measurements |
| $T$ | 2000 | Number of time steps. Metal GPU has a hard ~10s command-buffer timeout; T=2000 fits safely. On CPU you can use T=10000 |
| $D$ | 1 | State dimension. D=1 means the diagonal Jacobian in DEER is exact (no approximation) |
| $N$ | 256 | Number of particles. All $N$ trajectories are solved in parallel by DEER |

In [ ]:
PHI     = 0.9   # state transition coefficient
SIGMA_Q = 0.5   # process noise std
SIGMA_R = 1.0   # observation noise std

# Metal GPU has a hard ~10s command-buffer timeout; keep T small enough to stay under it.
# For CPU (jax.devices()[0].platform == 'cpu'), you can safely push T=10000.
T  = 2000  # number of timesteps  (original script used 10000)
D  = 1     # state dimension
N  = 256   # number of particles

## Step 1 — Kalman filter (exact posterior)

For a linear-Gaussian SSM the posterior $p(x_t \mid y_{1:t})$ is itself Gaussian and can be computed exactly by the Kalman filter. At each step $t$ it maintains a mean $m_t$ and variance $P_t$ and updates them via:

$$m_t^{\text{pred}} = \phi\, m_{t-1}, \qquad P_t^{\text{pred}} = \phi^2 P_{t-1} + \sigma_q^2$$

$$S_t = P_t^{\text{pred}} + \sigma_r^2 \qquad (\text{innovation variance})$$

$$K_t = P_t^{\text{pred}} / S_t \qquad (\text{Kalman gain})$$

$$m_t = m_t^{\text{pred}} + K_t (y_t - m_t^{\text{pred}}), \qquad P_t = (1 - K_t)\, P_t^{\text{pred}}$$

We use the KF output for two things:
1. **Ground truth** — the KF means are the optimal estimates; we measure RMSE of particle filter estimates relative to them.
2. **Twist initialisation** — the KF posterior $\mathcal{N}(m_t, P_t)$ is the *optimal* twist for a linear-Gaussian model (it encodes exactly the future observation information $p(y_{t:T} \mid x_t)$).

The implementation uses `jax.lax.scan` to run the $T$ update steps sequentially (this is just a standard Kalman recursion, not yet parallelized).

In [ ]:
def kalman_filter(ys, phi, sigma_q, sigma_r):
    def step(carry, y):
        m, P = carry
        m_pred = phi * m
        P_pred = phi**2 * P + sigma_q**2
        S = P_pred + sigma_r**2
        K = P_pred / S
        m_new = m_pred + K * (y - m_pred)
        P_new = (1.0 - K) * P_pred
        return (m_new, P_new), (m_new, P_new)
    _, (means, variances) = jax.lax.scan(step, (0.0, 1.0), ys)
    return means, variances

## Step 2 — Twisted proposals and importance weights

This is the conceptual heart of the algorithm. Read this carefully.

---

### Why standard particle filters fail at parallelism

A **bootstrap particle filter** draws proposals from the prior transition:
$$x_t^{(n)} \sim p(x_t \mid x_{t-1}^{(n)})$$
and weights each particle by the likelihood:
$$w_t^{(n)} \propto p(y_t \mid x_t^{(n)})$$

When observations are informative (small $\sigma_r$), the prior transition is a poor proposal — most particles land far from the observed $y_t$ and get near-zero weight. This is **weight collapse**. To prevent it, the filter **resamples**: it discards low-weight particles and duplicates high-weight ones.

Resampling is the fundamental obstacle to parallelism. Once you resample at time $t$, the particle identities at $t+1$ depend on all particle weights at $t$, which depend on all particle positions at $t$, and so on. Every timestep is coupled to every prior timestep — you cannot evaluate step $t+5$ until step $t+4$ is done.

---

### The twist function: definition

A **twist function** $\psi_t : \mathcal{X} \to \mathbb{R}_{>0}$ is an auxiliary non-negative function that approximates the **future evidence**:
$$\psi_t(x_t) \approx p(y_{t:T} \mid x_t)$$

That is, $\psi_t(x_t)$ is a scalar score that answers: *how likely is it that the future observations $y_t, y_{t+1}, \ldots, y_T$ are consistent with the latent state being $x_t$ right now?*

If we had the exact $\psi_t(x_t) = p(y_{t:T} \mid x_t)$, then the **twisted proposal**
$$q_t(x_t \mid x_{t-1}) \;\propto\; p(x_t \mid x_{t-1})\,\psi_t(x_t)$$
would be the *locally optimal* proposal — particles would be drawn directly toward regions consistent with all future data, not just the immediately prior state. In this ideal case the importance weights are exactly constant (all particles equally good), and **no resampling is needed at all**.

---

### Where twist functions come from: the backward information filter

The exact twist satisfies the backward recursion:
$$\psi_t^*(x_t) = p(y_{t:T} \mid x_t) = \int p(y_t \mid x_t)\, p(x_{t+1} \mid x_t)\, \psi_{t+1}^*(x_{t+1})\, dx_{t+1}$$
with terminal condition $\psi_T^*(x_T) = p(y_T \mid x_T)$.

This is the **backward information filter**. It is generally intractable — computing it exactly requires integrating over the full state space $\mathcal{X}$ at every step.

**For a linear-Gaussian model, the backward filter is the Kalman smoother**, and the exact twist $\psi_t^*(x_t)$ is itself a Gaussian:
$$\psi_t^*(x_t) = \mathcal{N}(x_t;\, m_t^{\text{KF}},\, P_t^{\text{KF}})$$
where $m_t^{\text{KF}}, P_t^{\text{KF}}$ are the Kalman filter posterior mean and variance. This is why we initialise the twist directly from the Kalman filter output — it is not an approximation for this model, it is exact.

For **non-linear models**, one uses an approximate backward filter: an extended Kalman filter, unscented transform, Gaussian process regression, or a learned network $\psi_\theta$. The quality of the twist determines how balanced the particle weights are and therefore how well parallelism works.

---

### The twisted proposal in closed form (Gaussian case)

We use a **Gaussian twist**: $\psi_t(x_t) = \mathcal{N}(x_t;\, \mu_t^\psi, (\tau_t^\psi)^2)$.

The twisted proposal is proportional to the product of two Gaussians:
$$q_t(x_t \mid x_{t-1}) \propto \underbrace{\mathcal{N}(x_t;\, \phi x_{t-1},\, \sigma_q^2)}_{\text{prior transition}} \cdot \underbrace{\mathcal{N}(x_t;\, \mu_t^\psi,\, (\tau_t^\psi)^2)}_{\text{twist}}$$

The product of two Gaussians is another Gaussian. Combining precisions (inverse variances):

$$\lambda_{\text{post}} = \underbrace{\frac{1}{\sigma_q^2}}_{\text{prior precision}} + \underbrace{\frac{1}{(\tau_t^\psi)^2}}_{\text{twist precision}}, \qquad \sigma^2_{\text{post}} = \frac{1}{\lambda_{\text{post}}}$$

$$\mu_{\text{post}} = \sigma^2_{\text{post}} \left(\frac{\phi\, x_{t-1}}{\sigma_q^2} + \frac{\mu_t^\psi}{(\tau_t^\psi)^2}\right)$$

The proposal mean $\mu_{\text{post}}$ is a **precision-weighted average** of where the dynamics say the state should be ($\phi x_{t-1}$) and where the twist (future data) says it should be ($\mu_t^\psi$). A stronger twist (smaller $\tau_t^\psi$) pulls the proposal further from the dynamics and closer to the observation signal.

This is exactly what `twisted_step_fn` samples from: $x_t \sim \mathcal{N}(\mu_{\text{post}},\, \sigma^2_{\text{post}})$.

---

### Why fixed noise enables DEER

`twisted_step_fn` draws from $\mathcal{N}(\mu_{\text{post}},\, \sigma^2_{\text{post}})$ using a pre-drawn standard normal `noise` from the tape:

```python
return mean_post + jnp.sqrt(var_post) * noise   # noise ~ N(0,1), fixed before the filter runs
```

This is the reparametrisation trick. With `noise` fixed, the step function becomes a **deterministic** map $x_t = f(x_{t-1})$. Four consequences:

1. The whole trajectory $x_{1:T}^{(n)}$ is a deterministic function of $x_0^{(n)}$
2. DEER can differentiate through it: $\partial x_t / \partial x_{t-1}$ exists and can be computed
3. Newton's method can be applied globally across all $T$ steps simultaneously
4. Each particle $n$ is completely independent — DEER solves all $N$ in parallel

Without fixed noise, sampling at each step breaks the Jacobian, breaks Newton, and breaks parallelism.

---

### Why this specific model is an ideal case for DEER

For $D=1$ with a Gaussian twist, `twisted_step_fn` is **affine** in $x_{t-1}$:

$$x_t = \underbrace{\sigma^2_{\text{post}}\, \frac{\phi}{\sigma_q^2}}_{J_t} x_{t-1} \;+\; \underbrace{\sigma^2_{\text{post}}\, \frac{\mu_t^\psi}{(\tau_t^\psi)^2} + \sigma_{\text{post}}\,\varepsilon_t^{(n)}}_{b_t^{(n)}}$$

The Jacobian $J_t = \sigma^2_{\text{post},t} / \sigma_q^2 \cdot \phi$ is a scalar that depends only on the twist parameters (not on $x$), and $b_t^{(n)}$ is a particle-specific constant. DEER's Newton step on this affine system converges **in a single iteration** — the linear recurrence is solved exactly in one parallel scan. This is why `iters` will typically print 1 or 2.

For a non-linear model, the Jacobian would depend on $x$ and multiple Newton iterations would be needed.

---

### The importance weight: correcting for the biased proposal

Because we sampled from $q_t$ rather than the true joint, we correct with an **incremental log importance weight**:

$$\log w_t^{(n)} = \underbrace{\log p(y_t \mid x_t^{(n)})}_{\text{likelihood}} + \underbrace{\log p(x_t^{(n)} \mid x_{t-1}^{(n)})}_{\text{prior transition}} - \underbrace{\log q_t(x_t^{(n)} \mid x_{t-1}^{(n)})}_{\text{proposal}}$$

- **Likelihood** rewards particles near the observation $y_t$
- **Prior transition** rewards particles that followed a probable dynamic path
- **Proposal** penalises particles that were actively guided there by the twist (corrects for the look-ahead advantage)

If the twist were perfect, these three terms cancel exactly and all weights are equal — zero variance, no resampling needed. In practice, residual weight variation is small and corrected in Step 8.

The **particle approximation** to the full smoothing distribution is:
$$p(x_t \mid y_{1:T}) \approx \sum_{n=1}^{N} W_T^{(n)}\, \delta_{x_t^{(n)}}, \qquad W_T^{(n)} \propto \prod_{s=1}^{T} w_s^{(n)}$$

---

### Summary

| Ingredient | Role |
|---|---|
| Twist $\psi_t$ | Encodes future-observation information; makes the proposal informed |
| Twisted proposal $q_t$ | Draws particles into high-likelihood regions without resampling |
| Fixed noise tape | Makes each trajectory a deterministic function of its initial state |
| Importance weights $w_t$ | Corrects the proposal bias; exact cancellation for a perfect twist |
| DEER | Solves all $T$ deterministic recurrences in $O(\log T)$ depth via Newton + associative scan |

In [ ]:
def twisted_step_fn(x_prev, driver, params):
    """
    Twisted transition for a single particle (D-dimensional).
    Combines prior N(phi*x, sigma_q^2) with Gaussian twist N(mu_twist, tau_twist^2)
    → Gaussian posterior in closed form.
    """
    obs, noise, twist = driver
    phi     = params['phi']
    sigma_q = params['sigma_q']

    mu_twist  = twist['mean']              # (D,)
    tau_twist = jnp.exp(twist['log_scale'])# (D,)

    prec_q     = 1.0 / sigma_q**2
    prec_twist = 1.0 / tau_twist**2
    var_post   = 1.0 / (prec_q + prec_twist)
    mean_post  = var_post * (prec_q * phi * x_prev + prec_twist * mu_twist)

    return mean_post + jnp.sqrt(var_post) * noise


def log_weight_fn(x_prev, x_next, obs, twist, params):
    """
    Incremental log importance weight:
        log w_t = log p(y_t|x_t) + log p(x_t|x_{t-1}) - log q_twist(x_t|x_{t-1})
    """
    phi     = params['phi']
    sigma_q = params['sigma_q']
    sigma_r = params['sigma_r']
    mu_twist  = twist['mean']
    tau_twist = jnp.exp(twist['log_scale'])

    log_lik = (-0.5 * jnp.sum((obs - x_next)**2) / sigma_r**2
               - 0.5 * D * jnp.log(2 * jnp.pi * sigma_r**2))

    log_trans = (-0.5 * jnp.sum((x_next - phi * x_prev)**2) / sigma_q**2
                 - 0.5 * D * jnp.log(2 * jnp.pi * sigma_q**2))

    prec_q     = 1.0 / sigma_q**2
    prec_twist = 1.0 / tau_twist**2
    var_post   = 1.0 / (prec_q + prec_twist)
    mean_post  = var_post * (prec_q * phi * x_prev + prec_twist * mu_twist)
    log_prop   = (-0.5 * jnp.sum((x_next - mean_post)**2 / var_post)
                  - 0.5 * jnp.sum(jnp.log(2 * jnp.pi * var_post)))

    return log_lik + log_trans - log_prop

## Step 3 — Simulate data

We sample a single realisation of the model:
- Draw $x_0 \sim \mathcal{N}(0, 1)$
- Run the state transition for $T$ steps using pre-drawn noise arrays
- Record observations $y_t = x_t + \sigma_r \eta_t$

`jax.lax.scan` folds the `sim_step` function over the noise arrays, propagating the state. The output `true_xs` is the ground-truth hidden trajectory we'll try to recover.

We reshape `ys` from `(T,)` to `(T, 1)` = `(T, D)` because the particle filter expects $D$-dimensional observations.

In [ ]:
master_key = jr.PRNGKey(0)
key, k1, k2, k3 = jr.split(master_key, 4)

x0 = jr.normal(k1) * 1.0

def sim_step(x, noise_obs):
    q_noise, r_noise = noise_obs
    x_new = PHI * x + SIGMA_Q * q_noise
    y     = x_new + SIGMA_R * r_noise
    return x_new, (x_new, y)

q_noise = jr.normal(k2, (T,))
r_noise = jr.normal(k3, (T,))
_, (true_xs, ys) = jax.lax.scan(sim_step, x0, (q_noise, r_noise))
ys_2d = ys[:, None]  # (T, 1)

print(f"Simulated T={T} timesteps,  true_xs shape: {true_xs.shape},  ys shape: {ys.shape}")

## Step 4 — Run the Kalman filter

We run the Kalman filter on the simulated observations to get:
- `kf_means[t]` — the posterior mean $\mathbb{E}[x_t \mid y_{1:t}]$ (the optimal point estimate)
- `kf_vars[t]` — the posterior variance $\text{Var}[x_t \mid y_{1:t}]$ (our uncertainty)

These values will double as the **twist parameters** in the next step: we set $\mu_t^\psi = m_t$ and $\tau_t^\psi = \sqrt{P_t}$, meaning the Kalman posterior is our look-ahead signal.

In [ ]:
kf_means, kf_vars = kalman_filter(ys, PHI, SIGMA_Q, SIGMA_R)
print(f"Kalman means range: [{kf_means.min():.3f}, {kf_means.max():.3f}]")

## Step 5 — Build the particle filter

### Twist parameters

We package the Kalman means and standard deviations into the twist dictionary. Shapes are `(T, D)` — one set of twist parameters per timestep per state dimension:

```python
twist_params['mean'][t]       = kf_means[t]           # μ_t^ψ
twist_params['log_scale'][t]  = 0.5 * log(kf_vars[t]) # log τ_t^ψ
```

For this linear-Gaussian model, the Kalman posterior *is* the optimal twist — it captures exactly the future information $p(y_{t:T} \mid x_t)$. In a non-linear model you'd compute an approximation instead.

### The "tape"

`tpf.make_tape(...)` pre-draws all $N \times T$ noise samples and packages them alongside the observations and twist parameters into a single array called the **tape**. This is crucial for DEER:

- DEER solves the recurrence $x_t = f(x_{t-1}, \varepsilon_t)$ by treating the noise $\varepsilon_t$ as a *fixed input* (not a random variable to resample)
- With fixed noise, the map $x_{t-1} \mapsto x_t$ is a deterministic function, so DEER can compute its Jacobian and apply Newton's method
- All $T$ steps are in the tape — DEER reads them all at once in parallel

### Initial particles

`x0s` shape is `(N, D)` — $N$ independent starting states, one per particle.

In [ ]:
twist_params = {
    'mean':      kf_means[:, None],                  # (T, D)
    'log_scale': 0.5 * jnp.log(kf_vars)[:, None],   # (T, D)
}

params = {
    'phi':     PHI,
    'sigma_q': SIGMA_Q,
    'sigma_r': SIGMA_R,
}

tpf = TwistedParallelParticleFilter(twisted_step_fn, log_weight_fn, D=D)

key, tape_key = jr.split(key)
tape = tpf.make_tape(ys_2d, twist_params, tape_key, N)

key, s0_key = jr.split(key)
x0s = jr.normal(s0_key, (N, D))

print(f"Tape created.  x0s shape: {x0s.shape}")

## Step 6 — Sequential twisted particle filter

This is the **baseline**: run the twisted PF the classical way, stepping through $t = 1, 2, \ldots, T$ one at a time using `jax.lax.scan`. Even though the proposal is twisted (informed), the $T$ steps are strictly sequential — each depends on the previous output.

- `run_seq` is JIT-compiled on the first call ("warmup"), then timed on the second
- Output `states_seq` has shape `(N, T, D)` — the full trajectory of all $N$ particles
- `seq_means` is the particle mean at each timestep: average over the $N$ particles

This gives us a reference time $t_{\text{seq}}$ to compare against the parallel version.

In [ ]:
run_seq = jax.jit(tpf.run_sequential)
_ = run_seq(x0s, tape, params)  # warmup

t0 = time.time()
states_seq = run_seq(x0s, tape, params).block_until_ready()
t_seq = time.time() - t0

seq_means = states_seq.mean(axis=0)[:, 0]  # (T,)
rmse_seq = jnp.sqrt(jnp.mean((seq_means - kf_means)**2))

print(f"Sequential twisted PF:  time={t_seq:.3f}s   RMSE vs Kalman={float(rmse_seq):.4f}")

## Step 7 — Parallel twisted particle filter via DEER

This is the **main result**.

### What DEER does

Each particle trajectory satisfies a discrete recurrence:
$$x_t^{(n)} = f(x_{t-1}^{(n)},\, \varepsilon_t^{(n)})$$

where $f$ is `twisted_step_fn` and the $\varepsilon_t^{(n)}$ are the pre-drawn tape values. DEER reformulates this as a root-finding problem:
$$g(\mathbf{x}^{(n)}) = 0 \qquad \text{where} \qquad g_t(\mathbf{x}) = x_t - f(x_{t-1}, \varepsilon_t)$$

It then applies **Newton's method** globally across all $T$ timesteps at once. Each Newton step linearises the system:
$$\Delta\mathbf{x} = -J^{-1} g(\mathbf{x})$$

where $J$ is the Jacobian of $g$ (a banded lower-triangular matrix). Solving this linear system is itself a sequential recurrence — but because the Jacobian is diagonal-block lower triangular, the solve decomposes into an **associative scan** (`jax.lax.associative_scan`) over $T$ steps, which JAX executes in $O(\log T)$ depth on GPU.

### Quasi-DEER: diagonal Jacobian approximation

The full Jacobian has $D^2 T$ entries. For $D=1$ this is exact and cheap. For larger $D$, the implementation uses only the **diagonal** of the Jacobian (quasi-DEER), which:
- Reduces memory from $O(D^2 T)$ to $O(DT)$
- Keeps each particle's recurrence independent (no cross-particle coupling)
- Is exact for $D=1$ (our case here)

### What you'll see

- `max_iter=50` — DEER runs up to 50 Newton iterations; it typically converges in 2–5
- `iters` — how many iterations DEER actually needed (usually very few for this model)
- The speedup comes from collapsing $O(T)$ sequential steps into $O(\log T)$ parallel depth

Note: `params['key']` gives DEER a random key for Hutchinson's estimator (used to estimate the diagonal Jacobian via random probing). For $D=1$ this is exact regardless.

In [ ]:
key, deer_key = jr.split(key)
params['key'] = deer_key

run_par = jax.jit(tpf.run_parallel, static_argnames=("max_iter", "num_samples"))
_ = run_par(x0s, tape, params, max_iter=50)  # warmup

t0 = time.time()
states_par, iters = run_par(x0s, tape, params, max_iter=50)
states_par.block_until_ready()
t_par = time.time() - t0

par_means  = states_par.mean(axis=0)[:, 0]  # (T,)
deer_iters = int(iters[0])
rmse_par   = jnp.sqrt(jnp.mean((par_means - kf_means)**2))

print(f"Parallel twisted PF:  time={t_par:.3f}s   RMSE vs Kalman={float(rmse_par):.4f}  ({deer_iters} DEER iters)")
print(f"Speedup: {t_seq / t_par:.2f}x")

## Step 8 — Importance-weighted means

The unweighted particle mean (`par_means` above) treats all $N$ particles equally. But because DEER initialises from `x0s` and converges to a *fixed point* of the twisted recurrence rather than exact samples from $q_t$, the particles may not be perfectly weighted.

We correct for this by computing **cumulative importance weights**:

$$W_t^{(n)} \propto \prod_{s=1}^{t} w_s^{(n)} = \exp\!\left(\sum_{s=1}^{t} \log w_s^{(n)}\right)$$

Everything is done in log-space for numerical stability:
1. `log_ws[n, t]` — the per-particle incremental log weight at each step
2. `cum_log_ws[n, t]` — cumulative sum → log of cumulative product of weights
3. `log_norm[t]` — log-sum-exp across particles → normalisation constant
4. `norm_ws[n, t]` — normalised weights, summing to 1 across particles at each $t$

The weighted mean $\hat{x}_t = \sum_n W_t^{(n)} x_t^{(n)}$ is a better estimate of $\mathbb{E}[x_t \mid y_{1:t}]$ than the unweighted mean, especially if some particles are more consistent with the data than others.

In [ ]:
log_ws = jax.jit(tpf.compute_log_weights)(x0s, states_par, tape, params)
cum_log_ws = jnp.cumsum(log_ws, axis=1)                          # (N, T)
log_norm   = jax.scipy.special.logsumexp(cum_log_ws, axis=0)     # (T,)
norm_ws    = jnp.exp(cum_log_ws - log_norm[None, :])             # (N, T)
weighted_par_means = jnp.sum(
    norm_ws[:, :, None] * states_par, axis=0
)[:, 0]  # (T,)

rmse_par_w = jnp.sqrt(jnp.mean((weighted_par_means - kf_means)**2))
print(f"Parallel PF (importance-weighted means):  RMSE vs Kalman={float(rmse_par_w):.4f}")

## Step 9 — Classical baselines

We compare against two standard particle-smoothing algorithms:

### Bootstrap particle filter (BPF)

The simplest particle filter:
1. **Propagate** — sample $x_t^{(n)} \sim p(x_t \mid x_{t-1}^{(n)})$ (from the prior, no look-ahead)
2. **Weight** — $w_t^{(n)} \propto p(y_t \mid x_t^{(n)})$
3. **Resample** — draw $N$ particles with replacement proportional to weights

The resampling step prevents weight collapse but is inherently sequential and collapses diversity. The output `bpf_filt` gives filtered means $\mathbb{E}[x_t \mid y_{1:t}]$ — slightly different from the smoother means $\mathbb{E}[x_t \mid y_{1:T}]$.

### Forward-filter backward-smoothing (FFBSi)

FFBSi is a **smoother** — it uses *all* observations $y_{1:T}$ (not just past ones) to refine each state estimate:
1. **Forward pass** — run the bootstrap PF to get filtered particles $\{x_t^{(n)}, w_t^{(n)}\}$
2. **Backward pass** — sample $M$ complete trajectories backwards, weighting by the transition density $p(x_{t+1}^{(m)} \mid x_t^{(n)})$

FFBSi is the gold-standard sequential smoother but has $O(MNT)$ cost — expensive. We use $M=N$ backward samples.

### What to compare

| Method | Uses future $y$? | Sequential depth | Cost |
|--------|-----------------|-----------------|------|
| Bootstrap PF | no (filter only) | $O(T)$ | $O(NT)$ |
| FFBSi | yes (smoother) | $O(T)$ | $O(MNT)$ |
| Twisted PF (sequential) | yes (via twist) | $O(T)$ | $O(NT)$ |
| **Twisted PF (DEER)** | **yes (via twist)** | **$O(\log T)$** | **$O(NT)$** |

In [ ]:
def lgssm_transition_mean(x_prev, p):
    return p['phi'] * x_prev

def lgssm_log_lik(y_t, x_t, p):
    return -0.5 * jnp.sum((y_t - x_t)**2) / p['sigma_r']**2

key, bpf_key, ffbsi_key, init_key = jr.split(key, 4)
x0_particles = jr.normal(init_key, (N, D)) * 1.0
bpf_params   = {'phi': PHI, 'sigma_q': SIGMA_Q, 'sigma_r': SIGMA_R}

run_bpf = jax.jit(bootstrap_pf, static_argnames=("transition_mean", "log_lik"))
_ = run_bpf(ys_2d, x0_particles, lgssm_transition_mean, lgssm_log_lik,
            SIGMA_Q, bpf_params, bpf_key)  # warmup

t0 = time.time()
bpf_filt, bpf_parts, bpf_log_w, bpf_ess, bpf_log_ml = run_bpf(
    ys_2d, x0_particles, lgssm_transition_mean, lgssm_log_lik,
    SIGMA_Q, bpf_params, bpf_key)
bpf_filt.block_until_ready()
t_bpf = time.time() - t0

rmse_bpf = float(jnp.sqrt(jnp.mean((bpf_filt[:, 0] - kf_means)**2)))
print(f"Bootstrap PF (filter):  time={t_bpf:.3f}s   RMSE vs Kalman={rmse_bpf:.4f}"
      f"  ESS mean={float(bpf_ess.mean()):.0f}/{N}")

In [ ]:
M_BACKWARD = N
run_ffbsi = jax.jit(ffbsi_smoother, static_argnames=("M", "transition_mean"))
_ = run_ffbsi(bpf_parts, bpf_log_w, lgssm_transition_mean,
              SIGMA_Q, bpf_params, ffbsi_key, M=M_BACKWARD)  # warmup

t0 = time.time()
bps_trajs = run_ffbsi(bpf_parts, bpf_log_w, lgssm_transition_mean,
                       SIGMA_Q, bpf_params, ffbsi_key, M=M_BACKWARD)
bps_trajs.block_until_ready()
t_bps = time.time() - t0

bps_means = bps_trajs.mean(axis=0)[:, 0]
rmse_bps  = float(jnp.sqrt(jnp.mean((bps_means - kf_means)**2)))
print(f"Bootstrap PS (FFBSi):   time={t_bps:.3f}s   RMSE vs Kalman={rmse_bps:.4f}  (M={M_BACKWARD})")
print(f"Twisted PF vs FFBSi:    RMSE  {float(rmse_par):.4f} vs {rmse_bps:.4f}"
      f"   time  {t_par:.3f}s vs {t_bps:.3f}s   speedup {t_bps/t_par:.1f}x")

## Results

The table below summarises all methods. Key things to look for:

- **RMSE** — how close each method's posterior mean is to the exact Kalman posterior mean. Lower is better. All particle-based smoothers should be close; the bootstrap *filter* will be slightly worse (it doesn't use future observations)
- **Time** — wall-clock seconds after JIT warmup
- **Speedup** — sequential twisted PF time divided by parallel twisted PF time. On Metal GPU this should be in the range 5–20× at T=2000
- **DEER iters** — how many Newton iterations DEER needed. For this linear-Gaussian model with a Kalman-derived twist, convergence is typically in 2–5 iterations

In [ ]:
print(f"T={T}, N={N} particles, D={D}")
print(f"{'Method':<35} {'Time (s)':>10} {'RMSE vs KF':>12}")
print("-" * 60)
print(f"{'Sequential twisted PF':<35} {t_seq:>10.3f} {float(rmse_seq):>12.4f}")
print(f"{'Parallel twisted PF (DEER)':<35} {t_par:>10.3f} {float(rmse_par):>12.4f}  ({deer_iters} iters)")
print(f"{'Parallel PF (weighted means)':<35} {'':>10} {float(rmse_par_w):>12.4f}")
print(f"{'Bootstrap PF (filter)':<35} {t_bpf:>10.3f} {rmse_bpf:>12.4f}")
print(f"{'Bootstrap PS (FFBSi)':<35} {t_bps:>10.3f} {rmse_bps:>12.4f}")
print(f"\nSpeedup (seq→parallel): {t_seq / t_par:.2f}x")
print(f"Speedup (FFBSi→parallel): {t_bps / t_par:.1f}x")

## Visualisation

Two panels:

**Top** — Sequential twisted PF vs Kalman. Observations (dots) are noisy; the Kalman mean (green) and sequential PF mean (blue dashed) should track the true state (black) closely.

**Bottom** — Parallel twisted PF (DEER) vs Kalman. The DEER estimate (red dashed) should be nearly indistinguishable from the sequential version — same particles, same twist, just solved in parallel. The title reports the wall-clock speedup.

If RMSE values are similar across the two panels, DEER has converged correctly. If the parallel mean looks noisier, increasing `max_iter` may help.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

t_idx = jnp.arange(T)

ax = axes[0]
ax.plot(t_idx, true_xs, "k-", lw=1,    label="true state", alpha=0.7)
ax.plot(t_idx, ys,      "k.", ms=2,    label="observations", alpha=0.3)
ax.plot(t_idx, kf_means, "g-", lw=1.5, label="Kalman mean")
ax.plot(t_idx, seq_means, "b--", lw=1.5,
        label=f"seq twisted PF (RMSE={float(rmse_seq):.3f})")
ax.legend(fontsize=9)
ax.set_title("Sequential twisted particle filter vs Kalman")

ax = axes[1]
ax.plot(t_idx, true_xs,  "k-", lw=1, alpha=0.7)
ax.plot(t_idx, kf_means, "g-", lw=1.5, label="Kalman mean")
ax.plot(t_idx, par_means, "r--", lw=1.5,
        label=f"parallel twisted PF ({deer_iters} iters, RMSE={float(rmse_par):.3f})")
ax.legend(fontsize=9)
ax.set_title(f"Parallel twisted PF (per-particle DEER) vs Kalman — {t_seq/t_par:.1f}x speedup")
ax.set_xlabel("timestep")

plt.tight_layout()
plt.savefig("twisted_pf_linear_gaussian.png", dpi=150)
plt.show()